In [4]:
import vertexai
from vertexai.generative_models import GenerativeModel, GenerationConfig
import json

class VertexCRNMutator:
    def __init__(self, project_id: str, location: str = "europe-west1"):
        """
        Args:
            project_id: Your Google Cloud Project ID (ask your IT admin if unsure).
            location: 'europe-west1' is Zurich (Fastest for you).
        """
        # 1. Initialize the Vertex SDK
        vertexai.init(project=project_id, location=location)
        
        # 2. Configure the Model
        # We use 'gemini-1.5-flash-001' for the loop because it's fast.
        # Use 'gemini-1.5-pro-001' if you need deep reasoning (slower).
        self.model = GenerativeModel("gemini-2.5-flash")
        
        # System instructions are set purely via context in Vertex currently, 
        # or implicitly in the prompt structure, though newer SDKs support system_instruction.
        self.system_prompt = """
        You are an expert Synthetic Biologist specialized in Mass Action Kinetics.
        You act as a mutation operator in an evolutionary algorithm.
        You ALWAYS output raw JSON. No markdown formatting.
        """

    def propose_mutation(self, current_crn_str: str, task_description: str):
        
        # 3. Define the detailed prompt
        prompt = f"""
        {self.system_prompt}

        TASK: {task_description}

        CURRENT CRN TOPOLOGY:
        {current_crn_str}

        OBJECTIVE:
        The current network is getting stuck in a local optimum.
        Suggest 2 structural changes (additions) to improve robustness.
        
        OUTPUT FORMAT (JSON ONLY):
        {{
            "reasoning": "One sentence explanation",
            "new_reactions": ["A + B -> C; [MAK(1.0)]", ...]
        }}
        """

        # 4. Configuration for JSON Mode
        # Vertex AI supports response_mime_type to force JSON
        config = GenerationConfig(
            temperature=0.8, # Slightly higher for "evolutionary exploration"
            response_mime_type="application/json" 
        )

        try:
            # 5. Call the API
            response = self.model.generate_content(
                prompt,
                generation_config=config
            )
            
            # 6. Parse
            # Vertex returns a candidate object
            text_response = response.text
            return json.loads(text_response)

        except Exception as e:
            print(f"Vertex AI Error: {e}")
            # In an EA, if the LLM fails, you usually fallback to random mutation
            return None

# --- Usage Example ---

# You must know your Project ID (it usually looks like "eth-research-group-12345")
PROJECT_ID = "crn-evolution" 

mutator = VertexCRNMutator(project_id=PROJECT_ID)

crn_data = """
Species: [X1, X2]
Rxns: X1 -> X2; [MAK(1.0)]
"""

result = mutator.propose_mutation(crn_data, "Create a toggle switch")

if result:
    print("AI Reasoning:", result['reasoning'])
    print("New Reactions:", result['new_reactions'])

AI Reasoning: To create a toggle switch and improve robustness, two reactions are added to establish mutual degradation between X1 and X2, forming the essential negative feedback loop for bistability.
New Reactions: ['X1 + X2 -> X1; [MAK(1.0)]', 'X2 + X1 -> X2; [MAK(1.0)]']


In [5]:
# now using the VertexCRNGenerator class 
from RL4CRN.NLPAgent.VertexCRNGenerator import VertexCRNGenerator

generator = VertexCRNGenerator(
    project_id=PROJECT_ID,
    location="europe-west1",
    model_name="gemini-2.5-flash"
)

hall_of_fame_texts = ["Loss::: 0.05509316434752127 CRN::: Inputs: ['u_1', 'u_2'] \nSpecies: ['X_1', 'Z_1', 'Z_2'] \nOutput Species: ['X_1'] \n∅ ----> Z_1;  [MAK(3.0, u_1)]\nX_1 ----> ∅;  [MAK(1.0, u_2)]\nX_1 ----> Z_2;  [MAK(0.3365837633609772)]\nX_1 ----> Z_2 + Z_2;  [MAK(0.33232152462005615)]\nZ_1 ----> X_1 + Z_1;  [MAK(0.38677549362182617)]\nX_1 + X_1 ----> ∅;  [MAK(0.33742326498031616)]\nZ_1 + Z_2 ----> X_1 + X_1;  [MAK(0.311252236366272)]",
 "Loss::: 0.05542411934287082 CRN::: Inputs: ['u_1', 'u_2'] \nSpecies: ['X_1', 'Z_1', 'Z_2'] \nOutput Species: ['X_1'] \n∅ ----> Z_1;  [MAK(3.0, u_1)]\nX_1 ----> ∅;  [MAK(1.0, u_2)]\nX_1 ----> Z_2;  [MAK(0.25462716817855835)]\nX_1 ----> Z_2 + Z_2;  [MAK(0.37102213501930237)]\nZ_1 ----> X_1 + Z_1;  [MAK(0.2930470108985901)]\nX_1 + X_1 ----> ∅;  [MAK(0.23316414654254913)]\nZ_1 + Z_2 ----> X_1;  [MAK(0.29570579528808594)]",
 "Loss::: 0.06588096411593304 CRN::: Inputs: ['u_1', 'u_2'] \nSpecies: ['X_1', 'Z_1', 'Z_2'] \nOutput Species: ['X_1'] \n∅ ----> Z_1;  [MAK(3.0, u_1)]\nX_1 ----> ∅;  [MAK(1.0, u_2)]\nX_1 ----> X_1 + Z_2;  [MAK(0.3485308885574341)]\nX_1 ----> Z_2 + Z_2;  [MAK(0.32132890820503235)]\nZ_1 ----> X_1 + Z_1;  [MAK(0.3853112459182739)]\nX_1 + X_1 ----> ∅;  [MAK(0.40606042742729187)]\nZ_1 + Z_2 ----> X_1 + X_1;  [MAK(0.3550707995891571)]",
 "Loss::: 0.06796756801686758 CRN::: Inputs: ['u_1', 'u_2'] \nSpecies: ['X_1', 'Z_1', 'Z_2'] \nOutput Species: ['X_1'] \n∅ ----> Z_1;  [MAK(3.0, u_1)]\nX_1 ----> ∅;  [MAK(1.0, u_2)]\nX_1 ----> Z_2;  [MAK(0.9885567426681519)]\nZ_1 ----> X_1 + Z_1;  [MAK(0.45307761430740356)]\nX_1 + X_1 ----> ∅;  [MAK(0.3914574384689331)]\nZ_1 + Z_2 ----> ∅;  [MAK(0.3385733366012573)]\nZ_1 + Z_2 ----> X_1 + X_1;  [MAK(0.20073792338371277)]",
 "Loss::: 0.06987618777683458 CRN::: Inputs: ['u_1', 'u_2'] \nSpecies: ['X_1', 'Z_1', 'Z_2'] \nOutput Species: ['X_1'] \n∅ ----> Z_1;  [MAK(3.0, u_1)]\nX_1 ----> ∅;  [MAK(1.0, u_2)]\nX_1 ----> Z_2;  [MAK(0.32803991436958313)]\nX_1 ----> Z_2 + Z_2;  [MAK(0.3365119695663452)]\nZ_1 ----> X_1 + Z_1;  [MAK(0.3237130045890808)]\nZ_1 + Z_2 ----> ∅;  [MAK(0.35611921548843384)]\nZ_1 + Z_2 ----> X_1 + X_1;  [MAK(0.34796783328056335)]",
 "Loss::: 0.07407437657754475 CRN::: Inputs: ['u_1', 'u_2'] \nSpecies: ['X_1', 'Z_1', 'Z_2'] \nOutput Species: ['X_1'] \n∅ ----> Z_1;  [MAK(3.0, u_1)]\nX_1 ----> ∅;  [MAK(1.0, u_2)]\nX_1 ----> Z_2;  [MAK(0.286340594291687)]\nX_1 ----> Z_2 + Z_2;  [MAK(0.3623330593109131)]\nZ_1 ----> X_1 + Z_1;  [MAK(0.48426634073257446)]\nX_1 + X_1 ----> ∅;  [MAK(0.2589777112007141)]\nZ_1 + Z_2 ----> ∅;  [MAK(0.3867933750152588)]",
 "Loss::: 0.07622712866096963 CRN::: Inputs: ['u_1', 'u_2'] \nSpecies: ['X_1', 'Z_1', 'Z_2'] \nOutput Species: ['X_1'] \n∅ ----> Z_1;  [MAK(3.0, u_1)]\nX_1 ----> ∅;  [MAK(1.0, u_2)]\nX_1 ----> Z_2 + Z_2;  [MAK(0.49247756600379944)]\nZ_1 ----> X_1 + Z_1;  [MAK(0.25740745663642883)]\nX_1 + X_1 ----> ∅;  [MAK(0.26527225971221924)]\nX_1 + Z_2 ----> Z_2;  [MAK(0.26713013648986816)]\nZ_1 + Z_2 ----> X_1 + X_1;  [MAK(0.17782452702522278)]",
 "Loss::: 0.08884000011500737 CRN::: Inputs: ['u_1', 'u_2'] \nSpecies: ['X_1', 'Z_1', 'Z_2'] \nOutput Species: ['X_1'] \n∅ ----> Z_1;  [MAK(3.0, u_1)]\nX_1 ----> ∅;  [MAK(1.0, u_2)]\nX_1 ----> Z_2;  [MAK(0.3683907985687256)]\nX_1 ----> Z_2 + Z_2;  [MAK(0.3080098032951355)]\nZ_1 ----> X_1 + Z_1;  [MAK(0.29498472809791565)]\nX_1 + Z_2 ----> Z_2;  [MAK(0.35458263754844666)]\nZ_1 + Z_2 ----> X_1 + X_1;  [MAK(0.3531334400177002)]",
 "Loss::: 0.09659711609881293 CRN::: Inputs: ['u_1', 'u_2'] \nSpecies: ['X_1', 'Z_1', 'Z_2'] \nOutput Species: ['X_1'] \n∅ ----> Z_1;  [MAK(3.0, u_1)]\nX_1 ----> ∅;  [MAK(1.0, u_2)]\nX_1 ----> Z_2;  [MAK(0.25448039174079895)]\nX_1 ----> Z_2 + Z_2;  [MAK(0.3781287968158722)]\nZ_1 ----> X_1 + Z_1;  [MAK(0.30003610253334045)]\nX_1 + X_1 ----> X_1;  [MAK(0.3760894238948822)]\nZ_1 + Z_2 ----> X_1 + X_1;  [MAK(0.22552095353603363)]",
 "Loss::: 0.11770502166178501 CRN::: Inputs: ['u_1', 'u_2'] \nSpecies: ['X_1', 'Z_1', 'Z_2'] \nOutput Species: ['X_1'] \n∅ ----> ∅;  [MAK(0.37762337923049927)]\n∅ ----> Z_1;  [MAK(3.0, u_1)]\nX_1 ----> ∅;  [MAK(1.0, u_2)]\nX_1 ----> Z_2 + Z_2;  [MAK(0.4852115213871002)]\nZ_1 ----> X_1 + Z_1;  [MAK(0.3579529821872711)]\nX_1 + X_1 ----> ∅;  [MAK(0.461677610874176)]\nZ_1 + Z_2 ----> X_1 + X_1;  [MAK(0.41780591011047363)]"]

list_of_allowed_reactions = 'Number of reactions: 91\nR0: ∅ ----> ∅;  [MAK(None)]\nR1: ∅ ----> X_1;  [MAK(None)]\nR2: ∅ ----> Z_1;  [MAK(None)]\nR3: ∅ ----> Z_2;  [MAK(None)]\nR4: ∅ ----> X_1 + X_1;  [MAK(None)]\nR5: ∅ ----> X_1 + Z_1;  [MAK(None)]\nR6: ∅ ----> X_1 + Z_2;  [MAK(None)]\nR7: ∅ ----> Z_1 + Z_1;  [MAK(None)]\nR8: ∅ ----> Z_1 + Z_2;  [MAK(None)]\nR9: ∅ ----> Z_2 + Z_2;  [MAK(None)]\nR10: X_1 ----> ∅;  [MAK(None)]\nR11: X_1 ----> Z_1;  [MAK(None)]\nR12: X_1 ----> Z_2;  [MAK(None)]\nR13: X_1 ----> X_1 + X_1;  [MAK(None)]\nR14: X_1 ----> X_1 + Z_1;  [MAK(None)]\nR15: X_1 ----> X_1 + Z_2;  [MAK(None)]\nR16: X_1 ----> Z_1 + Z_1;  [MAK(None)]\nR17: X_1 ----> Z_1 + Z_2;  [MAK(None)]\nR18: X_1 ----> Z_2 + Z_2;  [MAK(None)]\nR19: Z_1 ----> ∅;  [MAK(None)]\nR20: Z_1 ----> X_1;  [MAK(None)]\nR21: Z_1 ----> Z_2;  [MAK(None)]\nR22: Z_1 ----> X_1 + X_1;  [MAK(None)]\nR23: Z_1 ----> X_1 + Z_1;  [MAK(None)]\nR24: Z_1 ----> X_1 + Z_2;  [MAK(None)]\nR25: Z_1 ----> Z_1 + Z_1;  [MAK(None)]\nR26: Z_1 ----> Z_1 + Z_2;  [MAK(None)]\nR27: Z_1 ----> Z_2 + Z_2;  [MAK(None)]\nR28: Z_2 ----> ∅;  [MAK(None)]\nR29: Z_2 ----> X_1;  [MAK(None)]\nR30: Z_2 ----> Z_1;  [MAK(None)]\nR31: Z_2 ----> X_1 + X_1;  [MAK(None)]\nR32: Z_2 ----> X_1 + Z_1;  [MAK(None)]\nR33: Z_2 ----> X_1 + Z_2;  [MAK(None)]\nR34: Z_2 ----> Z_1 + Z_1;  [MAK(None)]\nR35: Z_2 ----> Z_1 + Z_2;  [MAK(None)]\nR36: Z_2 ----> Z_2 + Z_2;  [MAK(None)]\nR37: X_1 + X_1 ----> ∅;  [MAK(None)]\nR38: X_1 + X_1 ----> X_1;  [MAK(None)]\nR39: X_1 + X_1 ----> Z_1;  [MAK(None)]\nR40: X_1 + X_1 ----> Z_2;  [MAK(None)]\nR41: X_1 + X_1 ----> X_1 + Z_1;  [MAK(None)]\nR42: X_1 + X_1 ----> X_1 + Z_2;  [MAK(None)]\nR43: X_1 + X_1 ----> Z_1 + Z_1;  [MAK(None)]\nR44: X_1 + X_1 ----> Z_1 + Z_2;  [MAK(None)]\nR45: X_1 + X_1 ----> Z_2 + Z_2;  [MAK(None)]\nR46: X_1 + Z_1 ----> ∅;  [MAK(None)]\nR47: X_1 + Z_1 ----> X_1;  [MAK(None)]\nR48: X_1 + Z_1 ----> Z_1;  [MAK(None)]\nR49: X_1 + Z_1 ----> Z_2;  [MAK(None)]\nR50: X_1 + Z_1 ----> X_1 + X_1;  [MAK(None)]\nR51: X_1 + Z_1 ----> X_1 + Z_2;  [MAK(None)]\nR52: X_1 + Z_1 ----> Z_1 + Z_1;  [MAK(None)]\nR53: X_1 + Z_1 ----> Z_1 + Z_2;  [MAK(None)]\nR54: X_1 + Z_1 ----> Z_2 + Z_2;  [MAK(None)]\nR55: X_1 + Z_2 ----> ∅;  [MAK(None)]\nR56: X_1 + Z_2 ----> X_1;  [MAK(None)]\nR57: X_1 + Z_2 ----> Z_1;  [MAK(None)]\nR58: X_1 + Z_2 ----> Z_2;  [MAK(None)]\nR59: X_1 + Z_2 ----> X_1 + X_1;  [MAK(None)]\nR60: X_1 + Z_2 ----> X_1 + Z_1;  [MAK(None)]\nR61: X_1 + Z_2 ----> Z_1 + Z_1;  [MAK(None)]\nR62: X_1 + Z_2 ----> Z_1 + Z_2;  [MAK(None)]\nR63: X_1 + Z_2 ----> Z_2 + Z_2;  [MAK(None)]\nR64: Z_1 + Z_1 ----> ∅;  [MAK(None)]\nR65: Z_1 + Z_1 ----> X_1;  [MAK(None)]\nR66: Z_1 + Z_1 ----> Z_1;  [MAK(None)]\nR67: Z_1 + Z_1 ----> Z_2;  [MAK(None)]\nR68: Z_1 + Z_1 ----> X_1 + X_1;  [MAK(None)]\nR69: Z_1 + Z_1 ----> X_1 + Z_1;  [MAK(None)]\nR70: Z_1 + Z_1 ----> X_1 + Z_2;  [MAK(None)]\nR71: Z_1 + Z_1 ----> Z_1 + Z_2;  [MAK(None)]\nR72: Z_1 + Z_1 ----> Z_2 + Z_2;  [MAK(None)]\nR73: Z_1 + Z_2 ----> ∅;  [MAK(None)]\nR74: Z_1 + Z_2 ----> X_1;  [MAK(None)]\nR75: Z_1 + Z_2 ----> Z_1;  [MAK(None)]\nR76: Z_1 + Z_2 ----> Z_2;  [MAK(None)]\nR77: Z_1 + Z_2 ----> X_1 + X_1;  [MAK(None)]\nR78: Z_1 + Z_2 ----> X_1 + Z_1;  [MAK(None)]\nR79: Z_1 + Z_2 ----> X_1 + Z_2;  [MAK(None)]\nR80: Z_1 + Z_2 ----> Z_1 + Z_1;  [MAK(None)]\nR81: Z_1 + Z_2 ----> Z_2 + Z_2;  [MAK(None)]\nR82: Z_2 + Z_2 ----> ∅;  [MAK(None)]\nR83: Z_2 + Z_2 ----> X_1;  [MAK(None)]\nR84: Z_2 + Z_2 ----> Z_1;  [MAK(None)]\nR85: Z_2 + Z_2 ----> Z_2;  [MAK(None)]\nR86: Z_2 + Z_2 ----> X_1 + X_1;  [MAK(None)]\nR87: Z_2 + Z_2 ----> X_1 + Z_1;  [MAK(None)]\nR88: Z_2 + Z_2 ----> X_1 + Z_2;  [MAK(None)]\nR89: Z_2 + Z_2 ----> Z_1 + Z_1;  [MAK(None)]\nR90: Z_2 + Z_2 ----> Z_1 + Z_2;  [MAK(None)]'

template_crn_text = """Species: ['X_1', 'Z_1', 'Z_2']
Output Species: ['X_1']
∅ ----> Z_1;  [MAK(3.0, u_1)]
X_1 ----> ∅;  [MAK(1.0, u_2)]"""

candidates = generator.generate_candidates(
    hall_of_fame_texts, 
    "Construct a stochastic CRN that implements Robust Perfect Adaptation, tracking u_1 and rejecting disturbances from u_2. The network involves just three species, X_1 (output), Z_1, and Z_2. You must pick exactly 5 additional reactions starting from this template: {template_crn_text}", 
    list_of_allowed_reactions,
    num_candidates = 5)





/local0/rossin/git/CRN-GenerativeAI/.venv/lib/python3.10/site-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


In [6]:
candidates

[{'reasoning': "This CRN builds on the standard integral feedback motif. `X_1` produces `Z_2` (R12), `Z_1` catalyzes `X_1` production (R23). The integral action is implemented by `Z_2` degrading `Z_1` (R73), which is a negative feedback on `X_1`'s production. Quadratic `X_1` degradation (R37) ensures stability, and `Z_2` degradation (R28) prevents its unbounded accumulation. This structure combines strong negative feedback with self-regulation.",
  'reaction_ids': [2, 10, 12, 23, 73, 37, 28],
  'parameter_values': [[3.0], [1.0], [0.3], [0.4], [0.2], [0.35], [0.1]]},
 {'reasoning': "This design utilizes the `Z_1`/`Z_2` complex to produce `X_1` (R77), a common motif for error correction in Hall of Fame examples. `X_1` produces `Z_2` (R12) and `Z_1` catalyzes `X_1` (R23). Quadratic `X_1` degradation (R37) and `Z_2` degradation (R28) are included for robust behavior and stability. This variant emphasizes a synergistic production of `X_1` from the 'integrator' components.",
  'reaction_ids'